In [ ]:
w0, w1 = 1.0, 1.0
inputs = [1, 2, 3, 4, 5]
targets = [2, 3, 4, 5, 6]

epochs = 1254
N_relax = 100
step = 0.1
lr = 0.1

for epoch in range(epochs):
    for n in range(len(inputs)):
        x0 = inputs[n]
        x2 = targets[n]
        x1 = 0  # reset every example

        for t in range(N_relax):
            e1 = x1 - w0 * x0
            e2 = x2 - w1 * x1
            x1 = x1 + step * (-e1 + w1 * e2)

        e1 = x1 - w0 * x0
        e2 = x2 - w1 * x1
        w0 = w0 + lr * (e1 * x0)
        w1 = w1 + lr * (e2 * x1)

# energy on last example only (after last weight step)
E = 0.5 * ((x1 - w0 * x0) ** 2 + (x2 - w1 * x1) ** 2)
print(w0, w1, x1, E)

0.2731307020437934 4.317057965536442 1.3773046934739972 0.0015310529362762267


In [85]:
import torch

L = 4
w = torch.ones(L - 1, requires_grad=True)

inputs = [1, 2, 3, 4, 5]
targets = [2, 3, 4, 5, 6]

epochs = 10000
N_relax = 100
step = 0.1
lr = 0.1

for epoch in range(epochs):
    for n in range(len(inputs)):
        x0 = inputs[n]
        x_h = torch.zeros(2, requires_grad=True)
        x2 = targets[n]

        for i in range(N_relax):
            # E = 0.5 * ((x2 - w[1] * x1) ** 2 + (x1 - w[0] * x0) ** 2)
            E = 0.5 * (
                (x_h[0] - w[0] * x0) ** 2
                + (x_h[1] - w[1] * x_h[0]) ** 2
                + (x2 - w[2] * x_h[1]) ** 2
            )
            E.backward()
            with torch.no_grad():
                x_h -= step * x_h.grad
            x_h.grad.zero_()

        w.grad.zero_()
        E = 0.5 * (
            (x_h[0] - w[0] * x0) ** 2
            + (x_h[1] - w[1] * x_h[0]) ** 2
            + (x2 - w[2] * x_h[1]) ** 2
        )
        E.backward()
        w.data = w.data - lr * w.grad

E = 0.5 * (
    (x_h[0] - w[0] * x0) ** 2
    + (x_h[1] - w[1] * x_h[0]) ** 2
    + (x2 - w[2] * x_h[1]) ** 2
)
print("Input-to-hidden weight:      ", w[0].data)
print("Hidden-to-output weight:     ", w[1].data)
print("Activity of a hidden neuron: ", x_h.data)
print("Hidden-to-output weight:     ", E)

Input-to-hidden weight:       tensor(nan)
Hidden-to-output weight:      tensor(nan)
Activity of a hidden neuron:  tensor([nan, nan])
Hidden-to-output weight:      tensor(nan, grad_fn=<MulBackward0>)
